# Data Handling

**Pandas** is the most popular Python library for data manipulation and analysis. It provides two primary data structures:
- **Series**: A 1-dimensional labeled array (like a single column in Excel).
- **DataFrame**: A 2-dimensional labeled data structure (like an entire Excel spreadsheet or SQL table).

## 1. Loading Data

Usually, you load data using `pd.read_csv('filename.csv')` or `pd.read_excel('filename.xlsx')`. 

For this tutorial, we will use Python's `io` module to simulate loading a CSV file directly from a string so this notebook is fully self-contained.

In [2]:
import pandas as pd
import numpy as np
import io

# Simulating a CSV file
csv_data = """employee_id,name,department,salary,age,joined_date
101,Alice,Engineering,95000,28,2021-03-15
102,Bob,Sales,62000,34,2020-11-01
103,Charlie,Engineering,105000,,2019-06-23
104,Diana,HR,58000,29,2022-01-10
105,Evan,Sales,70000,41,
106,Fiona,Engineering,98000,31,2021-08-05
"""

# Load data into a DataFrame
df = pd.read_csv(io.StringIO(csv_data))

# Display the DataFrame
df

,employee_id,name,department,salary,age,joined_date
0,101,Alice,Engineering,95000,28.0,2021-03-15
1,102,Bob,Sales,62000,34.0,2020-11-01
2,103,Charlie,Engineering,105000,NaN,2019-06-23
3,104,Diana,HR,58000,29.0,2022-01-10
4,105,Evan,Sales,70000,41.0,NaN
5,106,Fiona,Engineering,98000,31.0,2021-08-05


## 2. Inspecting the Data

Before manipulating data, you need to understand its shape, types, and summary statistics.

In [3]:
print("--- First 3 Rows ---")
display(df.head(3))

print("\n--- DataFrame Info (Notice the null values in 'age' and 'joined_date') ---")
df.info()

print("\n--- Summary Statistics for Numeric Columns ---")
display(df.describe())

--- First 3 Rows ---


,employee_id,name,department,salary,age,joined_date
0,101,Alice,Engineering,95000,28.0,2021-03-15
1,102,Bob,Sales,62000,34.0,2020-11-01
2,103,Charlie,Engineering,105000,NaN,2019-06-23



--- DataFrame Info (Notice the null values in 'age' and 'joined_date') ---
<class 'pandas.DataFrame'>
RangeIndex: 6 entries, 0 to 5
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   employee_id  6 non-null      int64  
 1   name         6 non-null      str    
 2   department   6 non-null      str    
 3   salary       6 non-null      int64  
 4   age          5 non-null      float64
 5   joined_date  5 non-null      str    
dtypes: float64(1), int64(2), str(3)
memory usage: 420.0 bytes

--- Summary Statistics for Numeric Columns ---


,employee_id,salary,age
count,6.000000,6.000000,5.00000
mean,103.500000,81333.333333,32.60000
std,1.870829,20353.541870,5.22494
min,101.000000,58000.000000,28.00000
25%,102.250000,64000.000000,29.00000
50%,103.500000,82500.000000,31.00000
75%,104.750000,97250.000000,34.00000
max,106.000000,105000.000000,41.00000


## 3. Data Selection and Filtering

You can extract specific columns, rows, or filter data based on conditions using boolean indexing.

In [4]:
# Selecting a single column (returns a Pandas Series)
names = df['name']
print("Names Column:\n", names.values, "\n")

# Selecting multiple columns (pass a list of column names)
subset = df[['name', 'department']]
print("Subset DataFrame:\n", subset.head(2), "\n")

# Filtering using a condition (Boolean Indexing)
engineers = df[df['department'] == 'Engineering']
print("Only Engineers:\n", engineers, "\n")

# Multiple conditions (use & for AND, | for OR. Put conditions in parentheses!)
highly_paid_young = df[(df['salary'] > 80000) & (df['age'] < 30)]
print("Highly paid and under 30:\n", highly_paid_young)

Names Column:
 <StringArray>
['Alice', 'Bob', 'Charlie', 'Diana', 'Evan', 'Fiona']
Length: 6, dtype: str 

Subset DataFrame:
     name   department
0  Alice  Engineering
1    Bob        Sales 

Only Engineers:
    employee_id     name   department  salary   age joined_date
0          101    Alice  Engineering   95000  28.0  2021-03-15
2          103  Charlie  Engineering  105000   NaN  2019-06-23
5          106    Fiona  Engineering   98000  31.0  2021-08-05 

Highly paid and under 30:
    employee_id   name   department  salary   age joined_date
0          101  Alice  Engineering   95000  28.0  2021-03-15


## 4. Handling Missing Data (Cleaning)

Real-world data is messy. You'll often have `NaN` (Not a Number) values. You can either drop them or fill them with logical replacements.

In [5]:
# Check for missing values
print("Missing values per column:\n", df.isnull().sum(), "\n")

# STRATEGY 1: Fill missing values
# Let's fill the missing age with the average (mean) age of the company
mean_age = df['age'].mean()
df['age'] = df['age'].fillna(mean_age)
print(f"Filled missing ages with mean: {mean_age:.1f}")

# STRATEGY 2: Drop rows with missing values
# Let's drop anyone who doesn't have a 'joined_date'
df_clean = df.dropna(subset=['joined_date'])
print("\nDataFrame after dropping rows missing a joined_date:")
display(df_clean)

Missing values per column:
 employee_id    0
name           0
department     0
salary         0
age            1
joined_date    1
dtype: int64 

Filled missing ages with mean: 32.6

DataFrame after dropping rows missing a joined_date:


,employee_id,name,department,salary,age,joined_date
0,101,Alice,Engineering,95000,28.0,2021-03-15
1,102,Bob,Sales,62000,34.0,2020-11-01
2,103,Charlie,Engineering,105000,32.6,2019-06-23
3,104,Diana,HR,58000,29.0,2022-01-10
5,106,Fiona,Engineering,98000,31.0,2021-08-05


## 5. Data Aggregation (Group By)

The `.groupby()` method allows you to split your data into groups based on some criteria, apply a function to each group independently, and combine the results. It is identical to a SQL `GROUP BY`.

In [6]:
# Group by department and find the mean of the numeric columns
dept_stats = df_clean.groupby('department')[['salary', 'age']].mean()

print("Average Salary and Age by Department:")
display(dept_stats)

# Group by department and apply multiple different aggregation functions
advanced_stats = df_clean.groupby('department').agg({
    'salary': ['min', 'max', 'mean'],
    'employee_id': 'count'  # Count how many employees in each dept
})

print("\nAdvanced Aggregation:")
display(advanced_stats)

Average Salary and Age by Department:


,salary,age
department,,
Engineering,99333.333333,30.533333
HR,58000.000000,29.000000
Sales,62000.000000,34.000000



Advanced Aggregation:


salary                       employee_id
               min     max          mean       count
department                                          
Engineering  95000  105000  99333.333333           3
HR           58000   58000  58000.000000           1
Sales        62000   62000  62000.000000           1

## 6. Creating New Columns & Applying Functions

You can easily create new columns based on existing data, or use the `.apply()` method to run custom functions across your rows or columns.

In [7]:
# Simple math to create a new column
df_clean['monthly_salary'] = df_clean['salary'] / 12

# Using apply() with a lambda function
# Let's categorize employees based on salary
df_clean['salary_band'] = df_clean['salary'].apply(lambda x: 'High' if x > 80000 else 'Standard')

display(df_clean[['name', 'salary', 'monthly_salary', 'salary_band']])

,name,salary,monthly_salary,salary_band
0,Alice,95000,7916.666667,High
1,Bob,62000,5166.666667,Standard
2,Charlie,105000,8750.000000,High
3,Diana,58000,4833.333333,Standard
5,Fiona,98000,8166.666667,High
